# Task 2, Step 6: Visual Grounding Evaluation

Both final-test suites, with greedy decoding, thinking disabled, a 16-token cap, and batch size 8. The question and **original references stay fixed** across four conditions: correct image, complementary swap, mismatched image, and no image.

- **Grounding questions:** normalized exact match, separating presence and counting.
- **Natural VQAv2:** local soft consensus accuracy against all ten answers. These are local subset results, not server scores.
- **Pair accuracy:** both predictions match their canonical answers.
- **Intervals:** 95%, from 4,000 bootstrap resamples of complete pairs, matching pair IDs when computing differences.

Image sensitivity, correctness under the original image, and following the new image are different measurements. Teacher-forced likelihood and free generation also measure different behaviour.

In [ ]:
import os, sys, json
root = os.getcwd()
while not os.path.isdir(os.path.join(root, "task2_vlm_grounding")):
    root = os.path.dirname(root)
sys.path.insert(0, root)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

from task2_vlm_grounding.src.notebook_utils import *
setup_notebook()

In [ ]:
summary = load_final("summary")
by_type = load_final("by_type")
advantages = load_final("advantages")
suites = {"grounding": "Annotation-derived grounding", "natural": "Natural VQAv2"}
conditions = list(CONDITION_LABELS)
display(summary[["suite", "condition", "pairs", "score", "pair_correct", "identical", "empty", "follows_partner", "reference_nll", "seconds_per_answer"]].round(3))

## 6.1 Scores under the four image conditions

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
for ax, (suite, label) in zip(axes[:2], suites.items()):
    table = summary[summary["suite"] == suite].set_index("condition").loc[conditions]
    err = [table["score"] - table["score_low"], table["score_high"] - table["score"]]
    ax.bar([CONDITION_LABELS[c] for c in conditions], table["score"], yerr=err, capsize=4, color=[CONDITION_COLORS[c] for c in conditions], edgecolor="black")
    ax.set(title=label, ylabel="Exact match (%)" if suite == "grounding" else "Soft consensus accuracy (%)", ylim=(0, 100))
    ax.tick_params(axis="x", rotation=15)
pivot = by_type.pivot_table(index=["suite", "type"], columns="condition", values="score")[conditions].rename(columns=CONDITION_LABELS)
pivot.to_csv(f"{RESULTS_DIR}/final_by_question_type.csv")
sns.heatmap(pivot, annot=True, fmt=".1f", cmap="viridis", ax=axes[2], cbar_kws={"label": "%"})
axes[2].set(title="By question type", xlabel="", ylabel="")
axes[2].tick_params(axis="x", rotation=20)
save_figure("task2_6_conditions")

## 6.2 Pair diagnostics
A language-prior answer tends to give the same prediction for both images of a pair, so identical-answer rate rises under no-image inputs by construction.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 7.2), sharex=True)
for row, (suite, label) in zip(axes, suites.items()):
    table = summary[summary["suite"] == suite].set_index("condition").loc[conditions]
    for ax, metric, title in zip(row, ["pair_correct", "identical", "follows_partner"], ["Pair accuracy", "Identical-answer rate", "Prediction equals partner's answer"]):
        err = [table[metric] - table[f"{metric}_low"], table[f"{metric}_high"] - table[metric]]
        ax.bar([CONDITION_LABELS[c] for c in conditions], table[metric], yerr=err, capsize=4, color=[CONDITION_COLORS[c] for c in conditions], edgecolor="black")
        ax.set(title=f"{label}: {title}", ylabel="%", ylim=(0, 100))
        ax.tick_params(axis="x", rotation=15)
save_figure("task2_6_pair_diagnostics")

## 6.3 Correct-image advantages
Positive values favour the correct image. An interval crossing zero means this test set does not separate the conditions.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
labels = [f"{suites[s].split(' ')[0]}: vs. {CONDITION_LABELS[v]}" for s, v in zip(advantages["suite"], advantages["versus"])]
y = np.arange(len(advantages))[::-1]
for ax, mean, low, high, title in [(axes[0], "accuracy_advantage_pp", "accuracy_low", "accuracy_high", "Score advantage (percentage points)"), (axes[1], "nll_advantage", "nll_low", "nll_high", r"Reference NLL advantage ($\Delta$NLL)")]:
    ax.errorbar(advantages[mean], y, xerr=[advantages[mean] - advantages[low], advantages[high] - advantages[mean]], fmt="o", capsize=4, color="#2c3e50")
    ax.axvline(0, color="crimson", linestyle="--")
    ax.set_yticks(y); ax.set_yticklabels(labels if ax is axes[0] else [])
    ax.set(title=title)
save_figure("task2_6_advantages")
display(advantages.round(3))

## 6.4 Do swapped predictions follow the partner image?
Under a complementary swap, each prediction either keeps the original answer, moves to the partner's answer, or gives something unsupported.

In [ ]:
rows = []
for suite, label in suites.items():
    frame = pd.read_csv(f"{RESULTS_DIR}/main/final_{suite}_swap.csv", keep_default_na=False)
    original = frame["exact"].astype(float).mean() * 100
    partner = frame["follows_partner"].astype(float).mean() * 100
    rows.append({"Suite": label, "Kept original answer": original, "Followed partner image": partner, "Other": 100 - original - partner})
swap = pd.DataFrame(rows).set_index("Suite")
swap.to_csv(f"{RESULTS_DIR}/final_swap_behaviour.csv")
ax = swap.plot(kind="barh", stacked=True, figsize=(10, 2.8), color=["#e74c3c", "#2ecc71", "#95a5a6"], edgecolor="black")
ax.set(title="Prediction under the complementary swap", xlabel="% of question sides", ylabel="")
save_figure("task2_6_swap_behaviour")

## 6.5 Aligned complementary pairs
**Selection rule, fixed before viewing:** per suite, the first pair (by pair ID) in each outcome group under correct images: both right, one right, both wrong. Images show the original pair. Predictions come from all four conditions.

In [ ]:
for suite, label in suites.items():
    frames = {c: pd.read_csv(f"{RESULTS_DIR}/main/final_{suite}_{c}.csv", keep_default_na=False) for c in conditions}
    outcome = frames["correct"].groupby("pair_id")["exact"].sum()
    picks = [(name, outcome[outcome == value].index[0]) for name, value in [("Both right", 2), ("One right", 1), ("Both wrong", 0)] if (outcome == value).any()]
    fig, axes = plt.subplots(len(picks), 2, figsize=(11, 3.7 * len(picks)), squeeze=False)
    for row, (name, pair_id) in zip(axes, picks):
        for ax, side in zip(row, ["a", "b"]):
            line = frames["correct"][(frames["correct"]["pair_id"] == pair_id) & (frames["correct"]["side"] == side)].iloc[0]
            answers = " | ".join(f"{c}: {frames[c][(frames[c]['pair_id'] == pair_id) & (frames[c]['side'] == side)].iloc[0]['prediction'][:18]!r}" for c in conditions)
            show_image(ax, line["original_image"], f"{name} | {line['question'][:60]}\nReference: {line['target']}\n{answers}")
    fig.suptitle(label)
    save_figure(f"task2_6_cards_{suite}")